In [14]:
import json
import os 

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

from src.utils import *

%reload_ext autoreload
%autoreload 2

In [2]:
# Load data 

meta_path = r"meta_test.csv"
feats_path = r"feats_test.npy"

# download data for all possible MoAs
meta_test, feats_test = load_data(meta_path, feats_path)

# GYMNet eval

In [3]:
moa = ['Lipids', 'Respiration', 'Cytoskeleton', 'GPI', 'DMSO']


moa_dict = {moa[i] : int(i) for i in range(len(moa))}
meta_test['moa_coded'] = meta_test.apply(lambda x: moa_dict[x['MoA']], axis=1)

moa_decode = {float(i) : moa[i] for i in range(len(moa))}

In [4]:
from src.GYMNet_model import GYMData, GYMnet
# define the dataset and the model 
model_path = r"models\GYMNET"
weights_name = "model_20240904_222247_32_0.91"

ds = GYMData(feats_test, meta_test, nmoa=len(moa), l=1)

args = json.load(open(model_path + '/args.json'))

In [20]:
# load the model 
net = GYMnet(args['nfeat_1'], args['nfeat_2'], len(moa), dropout=args['drop'])

# load the weights
net.load_state_dict(torch.load(os.path.join(model_path, weights_name)))

C:\Users\GPGWX\AppData\Local\Temp\ipykernel_12796\4090510072.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load(os.path.join(model_path, weig

<All keys matched successfully>

In [21]:
# run model evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

net.to(device)

pred_class_all = []
true_class_all = []

meta_out_gcn = pd.DataFrame()

for i, data in tqdm(enumerate(ds)):
    inputs, adj, labels = data['feats'].to(device), data['adj_matrix'].to(device), data['moa'].to(device)

    outputs = net(inputs.unsqueeze(0), adj.unsqueeze(0))
    pred_class = torch.argmax(outputs, dim=1)
    pred_class = moa_decode[pred_class.item()]

    pred_class_all.append(pred_class)
    true_class_all.append(moa_decode[labels.item()])

meta_out_gcn['pred_moa'] = pred_class_all
meta_out_gcn['true_moa'] = true_class_all

1240it [00:29, 42.59it/s]


# Random forest eval 

In [11]:
# prepare the data 
meta_test_combined, feats_test_combined = combine_modality_features(meta_test, feats_test)
m, f = meta_test_combined, feats_test_combined
xtest, ytest = f, m.MoA

In [ ]:
# load the classifier

import joblib 
# load the model
clf = joblib.load(r'random_forest_model.pkl')

In [12]:
# make the prediction using the random forest
preds_class = clf.predict(xtest)
preds_proba = clf.predict_proba(xtest)

In [ ]:
majority = majority_vote_prediction(meta_test_combined, preds_class, preds_proba)